In [1]:
import pandas as pd
from openai import OpenAI
import os

In [2]:
api_key=os.environ["OPENAI_API_KEY"]

**1. creating documents and splitting**

In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter #This split by new line
from langchain_text_splitters import RecursiveCharacterTextSplitter  #This split by character not necessarily by new line i.e \n


#First example of creating and splitting document

doc1=[Document(page_content="South A frica is the country in Africa that is located on a south tip of Africa", metadata={"page":"1"}),
         Document(page_content="South A frica has nine provinces or states", metadata={"page":"2"}),
         Document(page_content="Mpumalanga is the best province because this is where Ndebeles stay", metadata={"page":"3"})]

doc_splitter= RecursiveCharacterTextSplitter(chunk_size=10, chunk_overlap=5)
chunks1=doc_splitter.split_documents(doc1)
print(chunks1)


[Document(metadata={'page': '1'}, page_content='South A'), Document(metadata={'page': '1'}, page_content='A frica'), Document(metadata={'page': '1'}, page_content='is the'), Document(metadata={'page': '1'}, page_content='country'), Document(metadata={'page': '1'}, page_content='in Africa'), Document(metadata={'page': '1'}, page_content='that is'), Document(metadata={'page': '1'}, page_content='located'), Document(metadata={'page': '1'}, page_content='on a'), Document(metadata={'page': '1'}, page_content='a south'), Document(metadata={'page': '1'}, page_content='tip of'), Document(metadata={'page': '1'}, page_content='of Africa'), Document(metadata={'page': '2'}, page_content='South A'), Document(metadata={'page': '2'}, page_content='A frica'), Document(metadata={'page': '2'}, page_content='has nine'), Document(metadata={'page': '2'}, page_content='provinces'), Document(metadata={'page': '2'}, page_content='or states'), Document(metadata={'page': '3'}, page_content='Mpumalanga'), Docume

In [4]:
#Second example of creating and splitting document

contents=["Kaizer Chiefs is the biggest football club in South Africa",
         "The Kaizer chiefs offices are based at Naturena in Gauteng province",
        "The founder of Kizer chiefs football club is Dr Boby Kaizer Motaung"]


doc2= [Document(page_content=content, metadata={"page":str(i+1)}) for i, content in enumerate(contents)]


doc_splitter2=RecursiveCharacterTextSplitter(chunk_size=20,chunk_overlap=10)
chunks2=doc_splitter2.split_documents(doc2)

print(chunks2)

[Document(metadata={'page': '1'}, page_content='Kaizer Chiefs is the'), Document(metadata={'page': '1'}, page_content='is the biggest'), Document(metadata={'page': '1'}, page_content='biggest football'), Document(metadata={'page': '1'}, page_content='football club in'), Document(metadata={'page': '1'}, page_content='club in South'), Document(metadata={'page': '1'}, page_content='in South Africa'), Document(metadata={'page': '2'}, page_content='The Kaizer chiefs'), Document(metadata={'page': '2'}, page_content='chiefs offices are'), Document(metadata={'page': '2'}, page_content='are based at'), Document(metadata={'page': '2'}, page_content='based at Naturena'), Document(metadata={'page': '2'}, page_content='Naturena in Gauteng'), Document(metadata={'page': '2'}, page_content='Gauteng province'), Document(metadata={'page': '3'}, page_content='The founder of Kizer'), Document(metadata={'page': '3'}, page_content='of Kizer chiefs'), Document(metadata={'page': '3'}, page_content='chiefs foo

In [5]:
#Third example of creating and splitting documents

df = pd.DataFrame({
    "Name": ["Lizy", "Methew", "Paul"],
    "gender": ["Female", "Female", "Male"]
})

doc3=[Document(page_content=", ".join(f"{col}: {row[col]}" for col in df.columns) , metadata={"row": i+1}) for i, row in df.iterrows()]

text_splitter3=CharacterTextSplitter(chunk_size=50, chunk_overlap=10)
chunks3=text_splitter3.split_documents(doc3)

print(chunks3)

[Document(metadata={'row': 1}, page_content='Name: Lizy, gender: Female'), Document(metadata={'row': 2}, page_content='Name: Methew, gender: Female'), Document(metadata={'row': 3}, page_content='Name: Paul, gender: Male')]


**2. Creating embeddings and Vectorstores**

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()  #convert text into numerical values, using openAI trained data for vector embeddings
vectorstore= FAISS.from_documents(chunks3,embeddings)
retriever= vectorstore.as_retriever()

**3. Creating a RAG Chain**

In [7]:
#This is building a RAG pipeline using LCEL (LangChain Expression Language),
#basically a clean, composable way to define how data flows through your system.

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

"""User Question
     ↓
Pass through + Retrieve context
     ↓
Insert into prompt
     ↓
LLM generates answer
     ↓
Convert to clean string"""

#Creatiing LLMs

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


prompt= PromptTemplate.from_template(
    """Answer the following question:
    {question}
    Based on the following context:
    {context}
    """
)

chain= (
    {"context": retriever, "question":RunnablePassthrough()}
    |prompt
    |llm
    |StrOutputParser()
)


**4. Creating a response**

In [8]:
response= chain.invoke("What gender is Lizzy?")
print(response)

Based on the provided context, Lizzy is female.


**5. Now we want to recall the history message**

In [9]:
from langchain_core.runnables import RunnableWithMessageHistory
#from langchain.memory import ChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

#Define a second prompt which accomodates chat history


"""prompt2 = ChatPromptTemplate.from_messages([
    ("You are a helpful assistant"),
    
    MessagesPlaceholder(variable_name="chat_history"),  # THIS is critical
    ("human", "{question}")
])"""


prompt2= PromptTemplate.from_template(
    
    """
    chatHistory:
    {chat_history}
    
    Answer the following question:
    {question}
    Based on the following context:
    {context}
 """   
)


#Using a new cjhain to accomaodate the chat history inside an input mapping
chain2 = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
        "chat_history": itemgetter("chat_history"),
    }
    | prompt2
    | llm
    | StrOutputParser()
)

#define a dictionary that will contain a message history

store={}

def get_message_history(session_id:str):
    if session_id not in store:
        store[session_id]= InMemoryChatMessageHistory()
    return store[session_id]


#wrap the chain2 with RunnableWithMessageHistory, to get chai with memory

chain_with_memory= RunnableWithMessageHistory(
    chain2,
    get_message_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

#Invoke with memory

response2 = chain_with_memory.invoke(
    {"question":"is it true that Methew is a male"},
    config={"configurable":{"session_id":"user1"}}
)

print(response2)

No, it is not true that Methew is a male. According to the provided context, Methew is identified as female.


In [10]:
response3 = chain_with_memory.invoke(
    {"question":"what question did I ask?"},
    config={"configurable":{"session_id":"user1"}}
)

print(response3)

You asked, "is it true that Methew is a male?"


In [13]:
leave=["quit","exit"]
while True:
    userInput=input("user: ")
    if userInput in leave:
        break

    response4 = chain_with_memory.invoke(
    {"question":userInput},
    config={"configurable":{"session_id":"user1"}}
    )

    print("assistant: \n",response4)

user:  exit


In [14]:
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import CharacterTextSplitter    #This split by new line
from langchain_text_splitters import RecursiveCharacterTextSplitter  #this split by character not necessarily by new line i.e \n
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

import os

# ---------------------------
# 2. Sample Documents
# ---------------------------
docs = [
    Document(page_content="LangChain is a framework for building applications with large language models.", metadata={"page":"1"}),
    Document(page_content="FAISS is a library for fast similarity search over vector embeddings.", metadata={"page":"2"}),
    Document(page_content="OpenAI provides powerful LLMs like GPT-4o-mini for reasoning and chat tasks.", metadata={"page":"3"}),
]



# ---------------------------
# 3. Split Documents
# ---------------------------
splitter = RecursiveCharacterTextSplitter(chunk_size=20, chunk_overlap=10)
chunks = splitter.split_documents(docs)


# ---------------------------
# 4. Embeddings + Vector Store
# ---------------------------
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever()


# ---------------------------
# 5. LLM
# ---------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ---------------------------
# 6. Prompt Template
# ---------------------------
prompt = PromptTemplate.from_template("""
You are a helpful assistant.

Use the context below to answer the question.

Context:
{context}

Question:
{question}

Answer clearly and concisely:
""")

# ---------------------------
# 7. RAG Chain (LCEL style)
# ---------------------------
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# ---------------------------
# 8. Ask a question
# ---------------------------
response = chain.invoke("What is the purpose of OpenAI?")
print(response)

The purpose of OpenAI is to provide capabilities for reasoning and chat tasks.


**Now do it to create dataframe and plots**

In [15]:
from langchain_core.runnables import RunnableWithMessageHistory
#from langchain.memory import ChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

#Define a second prompt which accomodates chat history


"""prompt2 = ChatPromptTemplate.from_messages([
    ("You are a helpful assistant"),
    
    MessagesPlaceholder(variable_name="chat_history"),  # THIS is critical
    ("human", "{question}")
])"""


prompt2= PromptTemplate.from_template(
    
    """
    chatHistory:
    {chat_history}
    
    Answer the following question:
    {question}
    Based on the following context:
    {context}
 """   
)


#Using a new cjhain to accomaodate the chat history inside an input mapping
chain2 = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
        "chat_history": itemgetter("chat_history"),
    }
    | prompt2
    | llm
    | StrOutputParser()
)

#define a dictionary that will contain a message history

store={}

def get_message_history(session_id:str):
    if session_id not in store:
        store[session_id]= InMemoryChatMessageHistory()
    return store[session_id]


#wrap the chain2 with RunnableWithMessageHistory, to get chai with memory

chain_with_memory= RunnableWithMessageHistory(
    chain2,
    get_message_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

#Invoke with memory

response2 = chain_with_memory.invoke(
    {"question":"is it true that Methew is a male"},
    config={"configurable":{"session_id":"user1"}}
)

print(response2)

Based on the provided context, there is no information regarding the gender of someone named Methew. Therefore, it cannot be determined if Methew is male or not.


In [16]:
#Invoke with memory

response11 = chain_with_memory.invoke(
    {"question":"what did I aske?"},
    config={"configurable":{"session_id":"user1"}}
)

print(response11)

You asked, "is it true that Methew is a male?"


In [18]:
docs = retriever.invoke("Methew gender")
for d in docs:
    print(d.page_content)

powerful LLMs like
for building
language models.
like GPT-4o-mini
